In [ ]:
from langgraph.graph import StateGraph
from langgraph.graph import StateGraph, START, END
from IPython.display import Image
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field, Literal
from langchain_core.messages import SystemMessage, HumanMessage
import operator

class BMIState(TypeDict):
  weight_kg: float
  height_kg: float
  bmi: float
  category: str

def calculate_bmi(state: BMIState) -> BMIState:
  weight = state['weight_kg']
  height = state['height_kg']
  bmi = weight / (height ** 2)
  state['bmi'] = round(bmi, 2)

  return state

def label_bmi(state: BMIState) -> BMIState:
  bmi = state['bmi']

  if bmi < 18.5:
      state["category"] = "Underweight"
  elif 18.5 <= bmi < 25:
      state["category"] = "Normal"
  elif 25 <= bmi < 30:
      state["category"] = "Overweight"
  else:
      state["category"] = "Obese"

  return state

graph = StateGraph(BMIState)
graph.add_node('calculate_bmi', calculate_bmi)
graph.add_node('label_bmi', label_bmi)
graph.add_edge(START, 'calculate_bmi')
graph.add_edge('calculate_bmi', 'label_bmi')
graph.add_edge('label_bmi', END)

workflow = graph.compile()
initial_state = {'weight_kg': 70, 'height_kg': 1.75}
final_state = workflow.invoke(initial_state)
print(final_state)
Image(workflow.get_graph().draw_mermaid_png())

In [ ]:
load_dotenv()
model = ChatOpenAI()

class LLMState(TypeDict):
  question: str
  answer: str

def llm_qa(state: LLMState) -> LLMState:
  question = state['question']
  prompt = f'Answer the following question {question}'
  answer = model.invoke(prompt).content
  state['answer'] = answer

  return state

graph = StateGraph(LLMState)
graph.add_node('llm_qa', llm_qa)
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

workflow = graph.compile()
initial_state = {'question': 'How far is moon from the earth?'}
final_state = workflow.invoke(initial_state)
print(final_state['answer'])
model.invoke('How far is moon from the earth?').content

In [ ]:
class BlogState(TypeDict):
  title: str
  outline: str
  content: str

def create_outline(state: BlogState) -> BlogState:
  title = state['title']
  prompt = f'Create an outline for a blog post about {title}'
  outline = model.invoke(prompt).content
  state['outline'] = outline

  return state

def create_content(state: BlogState) -> BlogState:
  title = state['title']
  outline = state['outline']
  prompt = f'Write a detailed blog on the title - {title} using the follwing outline \n {outline}'
  content = model.invoke(prompt).content
  state['content'] = content

  return state

graph = StateGraph(BlogState)
graph.add_node('create_outline', create_outline)
graph.add_node('create_content', create_content)
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_content')
graph.add_edge('create_content', END)

workflow = graph.compile()
initial_state = {'title': 'Rise of AI in India'}
final_state = workflow.invoke(initial_state)
print(final_state)
print(final_state['outline'])
print(final_state['content'])

In [ ]:
class BatsmanState(TypeDict):
  runs: int
  balls: int
  fours: int
  sixes: int

  sr: float
  bpb: float
  boundary_percent: float
  summary: str

def calculate_sr(state: BatsmanState) -> BatsmanState:
  sr = (state['runs'] / state['balls']) * 100
  return {'sr': sr}

def calculate_bpb(state: BatsmanState) -> BatsmanState:
  bpb = state['balls'] / (state['fours'] + state['sixes'])
  return {'bpb': bpb}

def calculate_boundary_percent(state: BatsmanState) -> BatsmanState:
  boundary_percent = (((state['fours'] * 4) + (state['sixes'] * 6)) / state['runs']) * 100
  return {'boundary_percent': boundary_percent}

def summary(state: BatsmanState):
  summary = f"""
   Strike Rate - {state['sr']} \n
   Balls per boundary - {state['bpb']} \n
   Boundary percent - {state['boundary_percent']}
  """

  return {'summary': summary}

graph = StateGraph(BatsmanState)

graph.add_node('calculate_sr', calculate_sr)
graph.add_node('calculate_bpb', calculate_bpb)
graph.add_node('calculate_boundary_percent', calculate_boundary_percent)
graph.add_node('summary', summary)

graph.add_edge(START, 'calculate_sr')
graph.add_edge(START, 'calculate_bpb')
graph.add_edge(START, 'calculate_boundary_percent')

graph.add_edge('calculate_sr', 'summary')
graph.add_edge('calculate_bpb', 'summary')
graph.add_edge('calculate_boundary_percent', 'summary')

graph.add_edge('summary', END)
workflow = graph.compile()
workflow

In [ ]:
intial_state = {
    'runs': 100,
    'balls': 50,
    'fours': 6,
    'sixes': 4
}

workflow.invoke(intial_state)

In [ ]:
model = ChatOpenAI(model='gpt-4o-mini')

class EvaluationSchema(BaseModel):
  feedback: str = Field(description='Detailed feedbackfor the essay')
  score: str = Field(description='Score out of 10', ge=0, le=10)

structured_model = model.with_structured_output(EvaluationSchema)
essay = """India in the Age of AI
As the world enters a transformative era defined by artificial intelligence (AI), India stands at a critical juncture — one where it can either emerge as a global leader in AI innovation or risk falling behind in the technology race. The age of AI brings with it immense promise as well as unprecedented challenges, and how India navigates this landscape will shape its socio-economic and geopolitical future.

India's strengths in the AI domain are rooted in its vast pool of skilled engineers, a thriving IT industry, and a growing startup ecosystem. With over 5 million STEM graduates annually and a burgeoning base of AI researchers, India possesses the intellectual capital required to build cutting-edge AI systems. Institutions like IITs, IIITs, and IISc have begun fostering AI research, while private players such as TCS, Infosys, and Wipro are integrating AI into their global services. In 2020, the government launched the National AI Strategy (AI for All) with a focus on inclusive growth, aiming to leverage AI in healthcare, agriculture, education, and smart mobility.

One of the most promising applications of AI in India lies in agriculture, where predictive analytics can guide farmers on optimal sowing times, weather forecasts, and pest control. In healthcare, AI-powered diagnostics can help address India’s doctor-patient ratio crisis, particularly in rural areas. Educational platforms are increasingly using AI to personalize learning paths, while smart governance tools are helping improve public service delivery and fraud detection.

However, the path to AI-led growth is riddled with challenges. Chief among them is the digital divide. While metropolitan cities may embrace AI-driven solutions, rural India continues to struggle with basic internet access and digital literacy. The risk of job displacement due to automation also looms large, especially for low-skilled workers. Without effective skilling and re-skilling programs, AI could exacerbate existing socio-economic inequalities.

Another pressing concern is data privacy and ethics. As AI systems rely heavily on vast datasets, ensuring that personal data is used transparently and responsibly becomes vital. India is still shaping its data protection laws, and in the absence of a strong regulatory framework, AI systems may risk misuse or bias.

To harness AI responsibly, India must adopt a multi-stakeholder approach involving the government, academia, industry, and civil society. Policies should promote open datasets, encourage responsible innovation, and ensure ethical AI practices. There is also a need for international collaboration, particularly with countries leading in AI research, to gain strategic advantage and ensure interoperability in global systems.

India’s demographic dividend, when paired with responsible AI adoption, can unlock massive economic growth, improve governance, and uplift marginalized communities. But this vision will only materialize if AI is seen not merely as a tool for automation, but as an enabler of human-centered development.

In conclusion, India in the age of AI is a story in the making — one of opportunity, responsibility, and transformation. The decisions we make today will not just determine India’s AI trajectory, but also its future as an inclusive, equitable, and innovation-driven society."""

prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {essay}'
structured_model.invoke(prompt).feedback

In [ ]:
class UPSCState(TypeDict):
  essay: str
  language_feedback: str
  analysis_feedback: str
  clarity_feedback: str
  overall_feedback: str
  individual_scores: Annotated[list[int], operator.add]
  avg_score: float

def evaluate_language(state: UPSCState):
  prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
  output = structured_model.invoke(prompt)

  return {'language_feedback': output.feedback, 'individual_scores': [output.score]}

def evaluate_analysis(state: UPSCState):
  prompt = f'Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
  output = structured_model.invoke(prompt)

  return {'analysis_feedback': output.feedback, 'individual_scores': [output.score]}

def evaluate_thought(state: UPSCState):

  prompt = f'Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
  output = structured_model.invoke(prompt)

  return {'clarity_feedback': output.feedback, 'individual_scores': [output.score]}
def final_evaluation(state: UPSCState):

  prompt = f'Based on the following feedbacks create a summarized feedback \n language feedback - {state["language_feedback"]} \n depth of analysis feedback - {state["analysis_feedback"]} \n clarity of thought feedback - {state["clarity_feedback"]}'
  overall_feedback = model.invoke(prompt).content

  avg_score = sum(state['individual_scores'])/len(state['individual_scores'])
  return {'overall_feedback': overall_feedback, 'avg_score': avg_score}

graph = StateGraph(UPSCState)
graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)

graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')

graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')

graph.add_edge('final_evaluation', END)
workflow = graph.compile()
workflow

In [ ]:
essay2 = """India and AI Time

Now world change very fast because new tech call Artificial Intel… something (AI). India also want become big in this AI thing. If work hard, India can go top. But if no careful, India go back.

India have many good. We have smart student, many engine-ear, and good IT peoples. Big company like TCS, Infosys, Wipro already use AI. Government also do program “AI for All”. It want AI in farm, doctor place, school and transport.

In farm, AI help farmer know when to put seed, when rain come, how stop bug. In health, AI help doctor see sick early. In school, AI help student learn good. Government office use AI to find bad people and work fast.

But problem come also. First is many villager no have phone or internet. So AI not help them. Second, many people lose job because AI and machine do work. Poor people get more bad.

One more big problem is privacy. AI need big big data. Who take care? India still make data rule. If no strong rule, AI do bad.

India must all people together – govern, school, company and normal people. We teach AI and make sure AI not bad. Also talk to other country and learn from them.

If India use AI good way, we become strong, help poor and make better life. But if only rich use AI, and poor no get, then big bad thing happen.

So, in short, AI time in India have many hope and many danger. We must go right road. AI must help all people, not only some. Then India grow big and world say "good job India"."""

initial_state = [
    'essay': essay2
]
workflow.invoke(initial_state)

In [ ]:
class QuadState(TypedDict):
  a: int
  b: int
  c: int

  equation: str
  discriminant: float
  result: str

def show_equation(state: QuadState):
  equation = f'{state["a"]}x2{state["b"]}x{state["c"]}'
  return {'equation':equation}

def calculate_discriminant(state: QuadState):
  discriminant = state["b"]**2 - (4*state["a"]*state["c"])
  return {'discriminant': discriminant}

def real_roots(state: QuadState):
  root1 = (-state["b"] + state["discriminant"]**0.5)/(2*state["a"])
  root2 = (-state["b"] - state["discriminant"]**0.5)/(2*state["a"])

  result = f'The roots are {root1} and {root2}'
  return {'result': result}

def repeated_roots(state: QuadState):
  root = (-state["b"])/(2*state["a"])
  result = f'Only repeating root is {root}'
  return {'result': result}

def no_real_roots(state: QuadState):
  result = f'No real roots'
  return {'result': result}

def check_condition(state: QuadState) -> Literal["real_roots", "repeated_roots", "no_real_roots"]:
  if state['discriminant'] > 0:
    return "real_roots"
  elif state['discriminant'] == 0:
    return "repeated_roots"
  else:
    return "no_real_roots"

graph = StateGraph(QuadState)
graph.add_node('show_equation', show_equation)
graph.add_node('calculate_discriminant', calculate_discriminant)
graph.add_node('real_roots', real_roots)
graph.add_node('repeated_roots', repeated_roots)
graph.add_node('no_real_roots', no_real_roots)

graph.add_edge(START, 'show_equation')
graph.add_edge('show_equation', 'calculate_discriminant')
graph.add_conditional_edges('calculate_discriminant', check_condition)

graph.add_edge('real_roots', END)
graph.add_edge('repeated_roots', END)
graph.add_edge('no_real_roots', END)

workflow = graph.compile()
workflow

In [ ]:
initial_state = {
    'a': 2,
    'b': 4,
    'c': 2
}

workflow.invoke(initial_state)

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model='gpt-4o-mini')

class SentimentSchema(BaseModel):
  sentiment: Literal["positive", "negative"] = Field(description='Sentiment of the review')

class DiagnosisSchema(BaseModel):
  issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
  tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
  urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')

structured_model = model.with_structured_output(SentimentSchema)
structured_model2 = model.with_structured_output(DiagnosisSchema)
prompt = 'What is the sentiment of the following review - The software too good'
structured_model.invoke(prompt).sentiment

In [ ]:
class ReviewState(TypeDict):
  review: str
  sentiment: Literal["positive", "negative"]
  diagnosis: dict
  response: str

def find_sentiment(state: ReviewState):
  prompt = f'For the following review find out the sentiment \n {state["review"]}'
  sentiment = structured_model.invoke(prompt).sentiment

  return {'sentiment': sentiment}

def check_sentiment(state: ReviewState) -> Literal["positive_response", "run_diagnosis"]:
  if state['sentiment'] == 'positive':
    return 'positive_response'
  else:
    return 'run_diagnosis'

def positive_response(state: ReviewState):
    prompt = f"""Write a warm thank-you message in response to this review:
    \n\n\"{state['review']}\"\n
    Also, kindly ask the user to leave feedback on our website."""
    response = model.invoke(prompt).content

    return {'response': response}

def run_diagnosis(state: ReviewState):
  prompt = f"""Diagnose this negative review:\n\n{state['review']}\n"
    "Return issue_type, tone, and urgency.
  """
  response = structured_model2.invoke(prompt)

  return {'diagnosis': response.model_dump()}

def negative_response(state: ReviewState):
  diagnosis = state['diagnosis']
  prompt = f"""You are a support assistant.
    The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
    Write an empathetic, helpful resolution message.
    """
  response = model.invoke(prompt).content

  return {'response': response}

graph = StateGraph(ReviewState)

graph.add_node('find_sentiment', find_sentiment)
graph.add_node('positive_response', positive_response)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('negative_response', negative_response)

graph.add_edge(START, 'find_sentiment')
graph.add_conditional_edges('find_sentiment', check_sentiment)
graph.add_edge('positive_response', END)
graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response', END)

workflow = graph.compile()
workflow

In [ ]:
intial_state={
    'review': "I’ve been trying to log in for over an hour now, and the app keeps freezing on the authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality."
}
workflow.invoke(intial_state)

In [ ]:
generator_llm = ChatOpenAI(model='gpt-4o-mini')
evaluator_llm = ChatOpenAI(model='gpt-4o-mini')
optimizer_llm = ChatOpenAI(model='gpt-4o-mini')

class TweetEvaluation(BaseModel):
  evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final evaluation result.")
  feedback: str = Field(..., description="feedback for the tweet.")

structured_evaluator_llm = evaluator_llm.with_structured_output(TweetEvaluation)

In [ ]:
class TweetState(TypeDict):
  topic: str
  tweet: str
  evaluation: Literal["approved", "needs_improvement"]
  feedback: str
  iteration: int
  max_iteration: int

  tweet_history: Annotated[list[str], operator.add]
  feedback_history: Annotated[list[str], operator.add]

def generate_tweet(state: TweetState):
  messages = [
        SystemMessage(content="You are a funny and clever Twitter/X influencer."),
        HumanMessage(content=f"""
          Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

          Rules:
          - Do NOT use question-answer format.
          - Max 280 characters.
          - Use observational humor, irony, sarcasm, or cultural references.
          - Think in meme logic, punchlines, or relatable takes.
          - Use simple, day to day english
        """)
    ]
  response = generator_llm.invoke(messages).content
  return {'tweet': response, 'tweet_history': [response]}

def evaluate_tweet(state: TweetState):
    messages = [
    SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
    HumanMessage(content=f"""
      Evaluate the following tweet:

      Tweet: "{state['tweet']}"

      Use the criteria below to evaluate the tweet:

      1. Originality – Is this fresh, or have you seen it a hundred times before?
      2. Humor – Did it genuinely make you smile, laugh, or chuckle?
      3. Punchiness – Is it short, sharp, and scroll-stopping?
      4. Virality Potential – Would people retweet or share it?
      5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

      Auto-reject if:
      - It's written in question-answer format (e.g., "Why did..." or "What happens when...")
      - It exceeds 280 characters
      - It reads like a traditional setup-punchline joke
      - Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

     ### Respond ONLY in structured format:
     - evaluation: "approved" or "needs_improvement"
     - feedback: One paragraph explaining the strengths and weaknesses
    """)
  ]

  response = structured_evaluator_llm.invoke(messages)
  return {'evaluation':response.evaluation, 'feedback': response.feedback, 'feedback_history': [response.feedback]}


def optimize_tweet(state: TweetState):

    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
          Improve the tweet based on this feedback:
          "{state['feedback']}"

        Topic: "{state['topic']}"
        Original Tweet:
        {state['tweet']}

        Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
    """)
  ]

  response = optimizer_llm.invoke(messages).content
  iteration = state['iteration'] + 1
  return {'tweet': response, 'iteration': iteration, 'tweet_history': [response]}

def route_evaluation(state: TweetState):
    if state['evaluation'] == 'approved' or state['iteration'] >= state['max_iteration']:
        return 'approved'
    else:
        return 'needs_improvement'


graph = StateGraph(TweetState)
graph.add_node('generate', generate_tweet)
graph.add_node('evaluate', evaluate_tweet)
graph.add_node('optimize', optimize_tweet)

graph.add_edge(START, 'generate')
graph.add_edge('generate', 'evaluate')
graph.add_conditional_edges('evaluate', route_evaluation, {'approved': END, 'needs_improvement': 'optimize'})
graph.add_edge('optimize', 'evaluate')

workflow = graph.compile()
workflow

In [ ]:
initial_state = {
    "topic": "srhberhb",
    "iteration": 1,
    "max_iteration": 5
}
result = workflow.invoke(initial_state)

In [ ]:
for tweet in result['tweet_history']:
  print(tweet)